In [0]:

from pyspark.sql import functions as F

tables = [
    ("Bronze Listings", "bootcamp_students.shipp_bronze.lb_listings_history", "listing_id", False),
    ("Bronze Requests", "bootcamp_students.shipp_bronze.lb_requests_history", "request_id", False),
    ("Bronze Saved Items", "bootcamp_students.shipp_bronze.lb_saved_items_history", "saved_item_id", False),
    ("Bronze Agent Activity", "bootcamp_students.shipp_bronze.lb_agent_activity_history", "activity_id", False),
    ("Silver Listings", "bootcamp_students.shipp_silver.silver_listings", "listing_id", True),
    ("Silver Requests", "bootcamp_students.shipp_silver.silver_requests", "request_id", True),
    ("Silver Candidate Pairs", "bootcamp_students.shipp_silver.silver_candidate_pairs", "pair_id", True),
    ("Silver Routes", "bootcamp_students.shipp_silver.silver_routes", None, False),
    ("Gold Matches", "bootcamp_students.shipp_gold.gold_candidate_matches", None, False),
    ("Gold Search Documents", "bootcamp_students.shipp_gold.gold_listing_search_docs", "listing_id", True),
    ("Gold Analytics", "bootcamp_students.shipp_gold.gold_marketplace_metrics", "metric_key", True)
]

results = []

for label, name, key, unique in tables:
    if not spark.catalog.tableExists(name):
        results.append((label, "MISSING", None, None, None))
        continue

    df = spark.table(name)
    row_count = df.count()

    null_keys = (
        df.filter(F.col(key).isNull()).count()
        if key and key in df.columns else None
    )

    duplicate_keys = (
        df.groupBy(key).count().filter(F.col("count") > 1).count()
        if unique and key in df.columns else None
    )

    results.append((
        label,
        "PRESENT" if row_count > 0 else "EMPTY",
        row_count,
        null_keys,
        duplicate_keys
    ))

audit = spark.createDataFrame(
    results,
    "table_name string, status string, rows long, null_keys long, duplicate_keys long"
)
display(audit)
